# Create Health Research Board (Ireland) Awards

**HRB** — Health Research Board, Ireland (funder_id `4320312041`, ROR `003hb2249`,
DOI `10.13039/100010414`, priority `373`). Source: the HRB "Grants approved" register
(one static HTML page of 1,675 grant cards + per-grant detail pages; see
scripts/local/hrb_ireland_to_s3.py). **1,675 grants**, **100% title / PI / institution /
EUR amount / scheme**, 93% abstract. No native grant id on the pages → `funder_award_id`
synthesized from the detail-URL slug (`hrb:<slug>`). Years derived from the Award Year
(source carries only a relative Duration) → start_year set, start_date null. Existing
OpenAlex coverage was Crossref-derived only — this adds canonical project metadata.

Parquet: `s3://openalex-ingest/awards/hrb_ireland/hrb_ireland_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrb_ireland_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hrb_ireland/hrb_ireland_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.hrb_ireland_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hrb_ireland_raw LIMIT 5;

## Step 2: Create HRB Ireland Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrb_ireland_awards
USING delta
AS
WITH
hrb_funder AS (
    -- HRB F4320312041 in common.funder (ROR 003hb2249, DOI 10.13039/100010414)
    SELECT CAST(funder_id AS BIGINT) as funder_id, display_name, ror_id, doi
    FROM openalex.common.funder WHERE funder_id = 4320312041
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title), ''), CONCAT('HRB grant ', g.funder_award_id)) as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN TRY_CAST(g.amount AS DECIMAL(18,2)) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN g.currency ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'hrb_ireland' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd')) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'Ireland' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.hrb_ireland_raw g
    CROSS JOIN hrb_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw WHERE provenance = 'hrb_ireland' AND priority = 373;
INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id, amount, currency, funder, funding_type, funder_scheme, provenance, start_date, end_date, start_year, end_year, lead_investigator, co_lead_investigator, investigators, landing_page_url, doi, works_api_url, created_date, updated_date, 373 as priority
FROM openalex.awards.hrb_ireland_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(DISTINCT funder_award_id) uniq_award, COUNT(DISTINCT id) uniq_id, COUNT(DISTINCT funder_id) funders,
  SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(amount) has_amount, COUNT(lead_investigator) has_pi, COUNT(description) has_desc, COUNT(start_year) has_year,
  ROUND(SUM(amount)/1000000,1) total_meur, MIN(start_year) min_yr, MAX(start_year) max_yr
FROM openalex.awards.hrb_ireland_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt, ROUND(SUM(amount)/1000000,1) meur
FROM openalex.awards.hrb_ireland_awards GROUP BY 1 ORDER BY 2 DESC LIMIT 12;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='hrb_ireland' AND priority=373;